# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shile/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The task is ultimately a clustering task. Fot the diagnostic tool, it uses unsupervised machine learning in grouping low performance pages based on similar features which can be easily used in examining the possible cause of failure.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no real target here. Content-related data is used along with an engineered feature called perfomance score for the clustering. All performance metrics are eliminated from this final layer to prevent leakage because the performance score is like an aggregate of the these scores.

In [15]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [16]:
df = pd.read_csv('/content/info_data.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,days_with_impressions_rank,days_with_sessions_rank,ctr_rank,engagement_rate_rank,scroll_rate_rank,ai_traffic_pct_rank,avg_position_rank,performance_score,perfomance_label,trend_pct_rank
0,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.809370,0.602785,0.468755,0.356948,0.612860,0.468059,0.307948,53.005026,good,0.291453
1,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.809370,0.650218,0.527444,0.356948,0.779332,0.468059,0.108826,56.159524,good,0.264776
2,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.809370,0.875573,0.587525,0.356948,0.745197,0.468059,0.068849,63.934734,good,0.494752
3,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,0.127038,0.078271,0.214070,0.356948,0.184839,0.468059,0.752740,26.228918,not good,0.065569
4,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,0.502118,0.920279,0.527444,0.885263,0.537231,0.468059,0.060953,65.907300,good,0.282363


In [17]:
df.iloc[0]

,0
content_id,content_a1fb4e703a9e
client_id,client_4e07408562
search_volume,90.0
competition,0.01
competition_level,LOW
cpc,0.05
content_type,keyword article
main_intent,informational
word_count,2481.0
char_count,15562.0


In [18]:
performance_cols = ['impressions_90d', 'clicks_90d', 'pageviews_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct', 'avg_position', 'performance_score', 'trend_direction']

for col in df.columns:
  if '_rank' in col:
    performance_cols.append(col)

df = df.drop(columns = performance_cols)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13474 entries, 0 to 13473
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              13474 non-null  object 
 1   client_id               13474 non-null  object 
 2   search_volume           13366 non-null  float64
 3   competition             13366 non-null  float64
 4   competition_level       13366 non-null  object 
 5   cpc                     13366 non-null  float64
 6   content_type            13474 non-null  object 
 7   main_intent             13474 non-null  object 
 8   word_count              10337 non-null  float64
 9   char_count              10337 non-null  float64
 10  provider_used           4250 non-null   object 
 11  model_used              11310 non-null  object 
 12  content_age_days        13474 non-null  int64  
 13  days_since_last_update  13474 non-null  int64  
 14  perfomance_label        13473 non-null

In [23]:
df = df.drop(columns = ['main_intent'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13474 entries, 0 to 13473
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              13474 non-null  object 
 1   client_id               13474 non-null  object 
 2   search_volume           13366 non-null  float64
 3   competition             13366 non-null  float64
 4   competition_level       13366 non-null  object 
 5   cpc                     13366 non-null  float64
 6   content_type            13474 non-null  object 
 7   word_count              10337 non-null  float64
 8   char_count              10337 non-null  float64
 9   provider_used           4250 non-null   object 
 10  model_used              11310 non-null  object 
 11  content_age_days        13474 non-null  int64  
 12  days_since_last_update  13474 non-null  int64  
 13  perfomance_label        13473 non-null  object 
dtypes: float64(5), int64(2), object(7)
mem

In [28]:
df.rename(columns = {'perfomance_label':'performance_label'}, inplace = True)
df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,word_count,char_count,provider_used,model_used,content_age_days,days_since_last_update,performance_label
0,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,2481.0,15562.0,NaN,gemini-3-flash-preview,445,25,good
1,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,3515.0,23643.0,NaN,gemini-2.5-flash,141,20,good
2,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,2803.0,17469.0,NaN,gemini-3-flash-preview,263,14,good
3,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,3059.0,20810.0,google,gemini-3-flash-preview,90,20,not good
4,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,3807.0,24228.0,google,gemini-3-flash-preview,90,20,good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13469,content_d10b8ea1be67,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,2716.0,20401.0,google,gemini-3-flash-preview,117,20,not good
13470,content_8223440cd40c,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,5644.0,35926.0,NaN,gemini-3-flash-preview,286,104,good
13471,content_2d515c8f9fe1,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,4647.0,29856.0,NaN,gemini-2.5-flash,147,20,good
13472,content_e49e368bdeb2,client_d029fa3a95,0.0,0.00,LOW,0.00,comparison article,4391.0,31431.0,NaN,gemini-2.5-flash,221,20,good


In [31]:
df.shape

(13474, 14)

In [32]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'word_count', 'char_count',
       'provider_used', 'model_used', 'content_age_days',
       'days_since_last_update', 'performance_label'],
      dtype='object')

In [29]:
X = df.drop(columns = ['performance_label'])
y = df['performance_label']

In [30]:
cat_cols = ['content_id', 'client_id', 'competition_level', 'content_type', 'provider_used', 'model_used']
num_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']

len(cat_cols) + len(num_cols)


13

In [35]:
for i in cat_cols:
  print(f'{i}: {X[i].nunique()}')
  print()

content_id: 13474

client_id: 31

competition_level: 3

content_type: 2

provider_used: 2

model_used: 5



In [37]:
for j in num_cols:
  print(f'{j}: {X[j].describe()}')
  print()

search_volume: count    13366.000000
mean       184.718689
std       1873.077525
min          0.000000
25%          0.000000
50%          0.000000
75%         20.000000
max      74000.000000
Name: search_volume, dtype: float64

competition: count    13366.000000
mean         0.063571
std          0.174628
min          0.000000
25%          0.000000
50%          0.000000
75%          0.010000
max          1.000000
Name: competition, dtype: float64

cpc: count    13366.000000
mean         0.305639
std          1.920319
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        100.360000
Name: cpc, dtype: float64

word_count: count    10337.000000
mean      3274.595240
std       1375.821327
min        424.000000
25%       2568.000000
50%       2941.000000
75%       3714.000000
max       9179.000000
Name: word_count, dtype: float64

char_count: count     10337.000000
mean      21639.463481
std        9337.192327
min        3143.000000
25%       1685

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.